In [1]:
import import_first
from common import deploy_utils
import pandas as pd

In [2]:
def calculate_cluster_metrics(vehicle_metrics: pd.DataFrame, clusters: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate mean N, Tn and Tc for each vehicle cluster.
    """

    df = vehicle_metrics.merge(clusters, on="vehicle_id", how="left", validate="one_to_one")

    if df["cluster"].isna().any():
        missing = df.loc[
            df["cluster"].isna(),
            "vehicle_id"
        ].tolist()

        raise ValueError(
            f"Vehicles without cluster assignment: {missing[:10]}"
        )

    result = (
        df
        .groupby("cluster")
        .agg(
            number_of_vehicles=("vehicle_id", "size"),
            mean_N=("N", "mean"),
            mean_Tn=("Tn", "mean"),
            mean_Tc=("Tc", "mean")
        )
        .sort_index()
    )

    return result

def calculate_satisfaction(cluster_metrics: pd.DataFrame, N_required: list[float], Tn_required: list[float], alpha: float) -> pd.DataFrame:
    """
    Reproduce the C satisfaction calculation:

        S_N  = min(N / N_required, 1)

        S_Tn = min(Tn_required / Tn, 1)

        S = alpha * S_N + (1-alpha) * S_Tn
    """

    result = cluster_metrics.copy()

    for cluster in result.index:

        k = int(cluster)

        mean_N = result.loc[ cluster, "mean_N" ]
        mean_Tn = result.loc[ cluster, "mean_Tn" ]

        N_req = N_required[k]
        Tn_req = Tn_required[k]

        if N_req <= 0:
            N_satisfaction = 1.0
        else:
            N_satisfaction = min( mean_N / N_req, 1.0)

        if mean_Tn <= 0:
            Tn_satisfaction = 0.0
        else:
            Tn_satisfaction = min( Tn_req / mean_Tn, 1.0)

        satisfaction = ( alpha * N_satisfaction + (1.0 - alpha) * Tn_satisfaction)

        result.loc[ cluster, "N_satisfaction" ] = N_satisfaction
        result.loc[ cluster, "Tn_satisfaction" ] = Tn_satisfaction
        result.loc[ cluster, "satisfaction" ] = satisfaction

    return result

### **Generating Baseline and random methods deployments and comparing with SA**

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# =========================================================
# Configuration
# =========================================================

AREA_COV_P = np.arange(2.5, 30.1, 2.5)

NRUNS_RNG = 30
RNG_SEEDS = range(1, NRUNS_RNG + 1)

ALPHA = 0.5
Y_LIM = (0.4, 1.05)

DATA_DIR = Path("..") / "SA" / "data"

C_AND_G_TRACE_PATH = DATA_DIR / "c_and_g_trace.csv"
V_TRACE_PATH = DATA_DIR / "v_trace_reduced.csv"

RESULTS_DIR = Path("results")
BASELINE_DIR = RESULTS_DIR / "baselines"
PLOT_DIR = RESULTS_DIR / "plots"

SCENARIOS = {
    "cell": {
        "trace": "c_and_g",
        "clusters": "vc_from_c",
        "N_required": [2.0, 1.0, 3.0],
        "Tn_required": [90.0, 120.0, 60.0],
    },

    "graph": {
        "trace": "c_and_g",
        "clusters": "vc_from_g",
        "N_required": [1.0, 2.0, 3.0],
        "Tn_required": [120.0, 90.0, 60.0],
    },

    "vehicle": {
        "trace": "vehicle",
        "clusters": "vc_from_v",
        "N_required": [3.0, 1.0, 2.0],
        "Tn_required": [60.0, 120.0, 90.0],
    },
}
TRACE_PATHS = { "c_and_g": C_AND_G_TRACE_PATH, "vehicle": V_TRACE_PATH, }

SA_RESULTS_DIR = Path("..") / "testSA"
# Existing SA result organization
SA_DIRS = {
    "cell": SA_RESULTS_DIR / "cell_graph_res",
    "graph": SA_RESULTS_DIR / "cell_graph_res",
    "cell_inverted": SA_RESULTS_DIR / "cell_graph_res_inverted",
    "graph_inverted": SA_RESULTS_DIR / "cell_graph_res_inverted",
    "vehicle": SA_RESULTS_DIR / "veic_res",
    "vehicle_inverted": SA_RESULTS_DIR / "veic_res_inverted",
}

# =========================================================
# Candidate Cells and priorities inversion 
# =========================================================

def invert_priorities(values: list[float]) -> list[float]:
    """
    Swap the minimum and maximum values while keeping the
    intermediate value in its original position.
    """

    result = list(values)

    i_min = int(np.argmin(result))
    i_max = int(np.argmax(result))

    result[i_min], result[i_max] = (result[i_max], result[i_min])

    return result

def get_requirements(scenario_name: str) -> tuple[list[float], list[float]]:

    base_name = scenario_name.removesuffix("_inverted")
    config = SCENARIOS[base_name]

    N_required = config["N_required"]
    Tn_required = config["Tn_required"]

    if scenario_name.endswith("_inverted"):
        N_required = invert_priorities(N_required)
        Tn_required = invert_priorities(Tn_required)

    return N_required, Tn_required

def get_candidate_cells(trace_df: pd.DataFrame) -> list[tuple[int, int]]:
    """
    Candidate cells used by dense and random deployment.
    """
    cells = trace_df[["x", "y"]].drop_duplicates().astype(int)
    return [ (int(x), int(y)) for x, y in cells.itertuples(index=False, name=None) ]

def get_dense_cells(trace_df: pd.DataFrame,) -> list[tuple[int, int]]:
    """
    Rank cells by accumulated dwell time.

    Highest-dwell cells are selected first.
    """

    scores = (
        trace_df
        .groupby(["x", "y"], as_index=False)["cell_dwell_time"]
        .sum()
        .sort_values(
            ["cell_dwell_time", "x", "y"],
            ascending=[False, True, True],
        )
    )

    return [ (int(x), int(y)) for x, y in scores[["x", "y"]].itertuples(index=False, name=None) ]

# =========================================================
# Deployment generation and Evaluation
# =========================================================

def write_deployment(path: Path, deployment: list[tuple[int, int]]) -> None:

    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w") as file:
        for x, y in deployment:
            file.write(f"{x};{y}\n")

def generate_deployments(dataset_name: str, trace_path: Path, rsu_values: np.ndarray) \
    -> tuple[
    dict[int, list[tuple[int, int]]],
    dict[int, dict[int, list[tuple[int, int]]]], ]:
    """
    Generate dense and random deployments once for a dataset.

    Returns:
        dense_deployments[rsus]
        random_deployments[rsus][seed]
    """

    trace_df = deploy_utils.Trace.read_trace(trace_path)

    candidate_cells = get_candidate_cells(trace_df)
    dense_cells = get_dense_cells(trace_df)

    if len(candidate_cells) == 0:
        raise ValueError(f"No candidate cells found in {trace_path}")

    max_rsus = int(max(rsu_values))
    if max_rsus > len(candidate_cells):
        raise ValueError(f"{dataset_name}: requested {max_rsus} RSUs, "
            f"but only {len(candidate_cells)} candidate cells exist.")

    dense_deployments = {}
    random_deployments = {}

    dense_root = BASELINE_DIR / dataset_name / "dense"
    random_root = BASELINE_DIR / dataset_name / "random"
    for rsus in rsu_values:

        rsus = int(rsus)

        # Write dense
        dense = dense_cells[:rsus]
        dense_deployments[rsus] = dense
        write_deployment(dense_root / f"rsu_{rsus:04d}.csv", dense)

        # Write random
        random_deployments[rsus] = {}
        for seed in RNG_SEEDS:

            rng = np.random.default_rng(seed)

            indices = rng.choice(len(candidate_cells), size=rsus, replace=False)
            deployment = [ candidate_cells[int(i)] for i in indices ]
            random_deployments[rsus][seed] = deployment

            write_deployment(random_root / f"rsu_{rsus:04d}" / f"seed_{seed:02d}.csv", deployment)

    return dense_deployments, random_deployments

def get_satisfaction_vector(satisfaction: pd.DataFrame) -> np.ndarray:
    """
    Extract c0/c1/c2 satisfaction values.
    """

    satisfaction = satisfaction.sort_index()
    values = satisfaction["satisfaction"].to_numpy(dtype=float)

    if len(values) != 3:
        raise ValueError(f"Expected 3 clusters, found {len(values)}.")

    return values

def evaluate_deployment(trace: pd.DataFrame, clusters: pd.DataFrame,
    deployment: list[tuple[int, int]], N_required: list[float], Tn_required: list[float]) \
        -> np.ndarray:
    """
    Evaluate one deployment using the existing functions.

    Returns a satisfaction array (S_c0, S_c1, S_c2)
    """

    vehicle_metrics = deploy_utils.Metrics.get_vehicles_N_Tn_Tc(
        trace=trace, rsus=deployment, grid_width=100,)
    cluster_metrics = calculate_cluster_metrics(
        vehicle_metrics=vehicle_metrics, clusters=clusters)
    satisfaction = calculate_satisfaction(
        cluster_metrics=cluster_metrics,
        N_required=N_required, Tn_required=Tn_required, alpha=ALPHA)

    return get_satisfaction_vector(satisfaction)

# =========================================================
# SA raw results
# =========================================================

def read_sa_raw(experiment_dir: Path, scenario: str) -> pd.DataFrame:
    """
    Read only tuning_raw.csv from directories named rsu_NNNN.

    Example:
        experiment_dir/
            rsu_0422/
                tuning_raw.csv
            rsu_2112/
                tuning_raw.csv
    """

    rows = []
    for rsu_dir in experiment_dir.glob("rsu_*"):

        if not rsu_dir.is_dir():
            continue

        try:
            rsus_from_dir = int(rsu_dir.name.removeprefix("rsu_"))
        except ValueError:
            continue

        raw_file = rsu_dir / "tuning_raw.csv"
        if not raw_file.is_file():
            continue

        df = pd.read_csv(raw_file, sep=";")

        required_columns = {
            "scenario",
            "N",
            "Tn",
            "rsus",
            "seed",
            "best_objective",
            "best_c0_s",
            "best_c1_s",
            "best_c2_s",
        }

        missing = required_columns - set(df.columns)

        if missing:
            raise ValueError(f"{raw_file} is missing: {sorted(missing)}")

        df = df[ df["scenario"].str.lower() == scenario.lower() ].copy()
        if df.empty:
            continue

        if not (df["rsus"] == rsus_from_dir).all():
            raise ValueError(f"RSU mismatch in {raw_file}")

        rows.append(df)

    if not rows:
        raise RuntimeError(f"No tuning_raw.csv data for {scenario} in {experiment_dir}")

    return pd.concat(rows, ignore_index=True)

def aggregate_sa(df: pd.DataFrame,) -> pd.DataFrame:

    return (
        df.groupby("rsus")
        .agg(
            c0_mean=("best_c0_s", "mean"),
            c0_std=("best_c0_s", "std"),

            c1_mean=("best_c1_s", "mean"),
            c1_std=("best_c1_s", "std"),

            c2_mean=("best_c2_s", "mean"),
            c2_std=("best_c2_s", "std"),

            runs=("seed", "count"),
        )
        .reset_index()
        .sort_values("rsus")
    )

# =========================================================
# Baseline evaluation
# =========================================================

def read_vehicle_clusters(filename: str) -> pd.DataFrame:
    """
    Expected format:

        vehicle_id;cluster
    """

    return pd.read_csv(filename, sep=";", header=None, names=["vehicle_id", "cluster"])

def evaluate_baselines(name: str,
    trace: pd.DataFrame, clusters: pd.DataFrame,
    rsu_values: np.ndarray,
    dense_deployments, random_deployments) -> tuple[pd.DataFrame, pd.DataFrame]:

    N_required, Tn_required = get_requirements(name)

    dense_rows = []
    random_rows = []

    # -----------------------------------------------------
    # Dense
    # -----------------------------------------------------

    for rsus in rsu_values:

        rsus = int(rsus)

        satisfaction = evaluate_deployment(
            trace, clusters,
            dense_deployments[rsus],
            N_required, Tn_required)

        dense_rows.append({
            "rsus": rsus,
            "c0_s": satisfaction[0],
            "c1_s": satisfaction[1],
            "c2_s": satisfaction[2],
        })

    # -----------------------------------------------------
    # Random
    # -----------------------------------------------------

    for rsus in rsu_values:

        rsus = int(rsus)

        for seed in RNG_SEEDS:

            satisfaction = evaluate_deployment(
                trace, clusters,
                random_deployments[rsus][seed],
                N_required, Tn_required)

            random_rows.append({
                "rsus": rsus,
                "seed": seed,
                "c0_s": satisfaction[0],
                "c1_s": satisfaction[1],
                "c2_s": satisfaction[2],
            })

    dense_df = pd.DataFrame(dense_rows)
    random_df = pd.DataFrame(random_rows)
    return dense_df, random_df

def aggregate_random(df: pd.DataFrame) -> pd.DataFrame:

    return (
        df.groupby("rsus")
        .agg(
            c0_mean=("c0_s", "mean"),
            c0_std=("c0_s", "std"),

            c1_mean=("c1_s", "mean"),
            c1_std=("c1_s", "std"),

            c2_mean=("c2_s", "mean"),
            c2_std=("c2_s", "std"),
        )
        .reset_index()
        .sort_values("rsus")
    )

# =========================================================
# Plotting
# =========================================================

def plot_mean_std( ax, x, mean, std, label, marker="o", linestyle="-", band_alpha=0.15,):
    ax.plot( x, mean, marker=marker, linestyle=linestyle, label=label,)

    # No meaningful band when there is only one run.
    if std.notna().any():
        ax.fill_between( x, mean - std, mean + std, alpha=band_alpha,)

def plot_sa_vs_dense( name: str, sa_df: pd.DataFrame, dense_df: pd.DataFrame, rsu_values: np.ndarray,) -> None:

    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 12,
        "figure.titlesize": 18,
    })

    sa_df = ( sa_df[sa_df["rsus"].isin(rsu_values)] .sort_values("rsus"))
    dense_df = ( dense_df[dense_df["rsus"].isin(rsu_values)] .sort_values("rsus"))

    fig, ax = plt.subplots(figsize=(7, 4))

    x = rsu_values
    coverage = AREA_COV_P[:len(x)]

    # -----------------------------------------------------
    # SA: mean ± std
    # -----------------------------------------------------

    plot_mean_std( ax, x, sa_df["c0_mean"], sa_df["c0_std"], label="SA: C0", marker="o", band_alpha=0.12,)
    plot_mean_std( ax, x, sa_df["c1_mean"], sa_df["c1_std"], label="SA: C1", marker="o", band_alpha=0.12,)
    plot_mean_std( ax, x, sa_df["c2_mean"], sa_df["c2_std"], label="SA: C2", marker="o", band_alpha=0.12,)

    # -----------------------------------------------------
    # Dense: deterministic, therefore no std
    # -----------------------------------------------------

    ax.plot( x, dense_df["c0_s"], marker="s", linestyle="--", label="Dense: C0",)
    ax.plot( x, dense_df["c1_s"], marker="s", linestyle="--", label="Dense: C1",)
    ax.plot( x, dense_df["c2_s"], marker="s", linestyle="--", label="Dense: C2",)

    ax.set_xticks(x)
    ax.set_xticklabels( [f"{p:g}%" for p in coverage])

    ax.set_xlabel("Covered area")
    ax.set_ylabel("Sat(k)")
    # ax.set_title( name.replace("_", " ").title() + " - SA vs Dense")

    ax.set_ylim(*Y_LIM)

    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")

    fig.tight_layout()

    fig.savefig( PLOT_DIR / f"{name}_SA_v_dense.png", dpi=600,)

    plt.close(fig)

def plot_sa_vs_random( name: str, sa_df: pd.DataFrame, random_df: pd.DataFrame, rsu_values: np.ndarray,) -> None:

    plt.rcParams.update({
        "font.size": 14,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 12,
        "figure.titlesize": 18,
    })

    random_agg = aggregate_random(random_df)

    # Make sure both are in the same RSU order.
    sa_df = ( sa_df[sa_df["rsus"].isin(rsu_values)] .sort_values("rsus"))
    random_agg = ( random_agg[random_agg["rsus"].isin(rsu_values)] .sort_values("rsus"))

    fig, ax = plt.subplots(figsize=(7, 4))

    x = rsu_values
    coverage = AREA_COV_P[:len(x)]

    # -----------------------------------------------------
    # SA
    # -----------------------------------------------------

    plot_mean_std( ax, x, sa_df["c0_mean"], sa_df["c0_std"], label="SA: C0", marker="o", band_alpha=0.12,)
    plot_mean_std( ax, x, sa_df["c1_mean"], sa_df["c1_std"], label="SA: C1", marker="o", band_alpha=0.12,)
    plot_mean_std( ax, x, sa_df["c2_mean"], sa_df["c2_std"], label="SA: C2", marker="o", band_alpha=0.12,)

    # -----------------------------------------------------
    # Random
    # -----------------------------------------------------

    plot_mean_std( ax, x, random_agg["c0_mean"], random_agg["c0_std"], label="Random: C0", marker="s", linestyle="--", band_alpha=0.12,)
    plot_mean_std( ax, x, random_agg["c1_mean"], random_agg["c1_std"], label="Random: C1", marker="s", linestyle="--", band_alpha=0.12,)
    plot_mean_std( ax, x, random_agg["c2_mean"], random_agg["c2_std"], label="Random: C2", marker="s", linestyle="--", band_alpha=0.12,)

    ax.set_xticks(x)
    ax.set_xticklabels( [f"{p:g}%" for p in coverage])

    ax.set_xlabel("Covered area")
    ax.set_ylabel("Sat(k)")
    # ax.set_title( name.replace("_", " ").title() + " - SA vs Random")

    ax.set_ylim(*Y_LIM)

    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right")

    fig.tight_layout()

    fig.savefig( PLOT_DIR / f"{name}_SA_v_rng.png", dpi=600,)

    plt.close(fig)

# =========================================================
# Main
# =========================================================

def main() -> None:

    BASELINE_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------------------
    # Load SA results and determine RSU values to use
    # -----------------------------------------------------

    sa_data = {}

    rsu_values_by_dataset = { "c_and_g": set(), "vehicle": set() }
    for name, experiment_dir in SA_DIRS.items():

        base_name = name.removesuffix("_inverted")

        sa_scenario = base_name

        raw = read_sa_raw(experiment_dir, sa_scenario)
        agg = aggregate_sa(raw)

        if len(agg) < len(AREA_COV_P):
            raise ValueError(
                f"{name}: only {len(agg)} RSU values found, "
                f"but {len(AREA_COV_P)} are required."
            )

        rsus = np.sort(agg["rsus"].unique()) [:len(AREA_COV_P)]

        agg = agg[agg["rsus"].isin(rsus)].copy()

        sa_data[name] = { "agg": agg, "rsus": rsus, }

        dataset_name = SCENARIOS[base_name]["trace"]

        rsu_values_by_dataset[dataset_name].update(int(r) for r in rsus)

    # -----------------------------------------------------
    # Load traces and generate deployments once per dataset
    # -----------------------------------------------------

    traces = {
        "c_and_g": deploy_utils.Trace.read_trace(file_path=C_AND_G_TRACE_PATH),
        "vehicle": deploy_utils.Trace.read_trace(file_path=V_TRACE_PATH),
    }

    deployments = {}
    for dataset_name, rsu_set in rsu_values_by_dataset.items():

        rsu_values = np.array(sorted(rsu_set), dtype=int)
        deployments[dataset_name] = generate_deployments(dataset_name, TRACE_PATHS[dataset_name], rsu_values)

    # -----------------------------------------------------
    # Evaluate each SA scenario against both baselines
    # -----------------------------------------------------

    for name, data in sa_data.items():

        base_name = name.removesuffix("_inverted")

        dataset_name = SCENARIOS[base_name]["trace"]

        cluster_file = DATA_DIR / f"{SCENARIOS[base_name]['clusters']}.csv"
        clusters = read_vehicle_clusters(filename=cluster_file)

        trace = traces[dataset_name]

        rsu_values = data["rsus"]

        dense_deployments, random_deployments = deployments[dataset_name]

        dense_df, random_df = evaluate_baselines(
            name, trace, clusters,
            rsu_values,
            dense_deployments, random_deployments)

        # Keep the generated evaluation results.
        dense_df.to_csv( BASELINE_DIR / f"{name}_dense.csv", sep=";", index=False,)
        random_df.to_csv( BASELINE_DIR / f"{name}_random.csv", sep=";", index=False,)

        # Plots
        plot_sa_vs_dense(name, data["agg"], dense_df, rsu_values,)
        plot_sa_vs_random(name, data["agg"], random_df, rsu_values,)

In [4]:
main()